In [4]:
import numpy as np
import matplotlib.pyplot as plt
import networkx as nx
import cvxpy as cp

In [5]:


# ==========================================
# 1. Συνάρτηση για Βελτιστοποίηση του λ2 (SDP)
# ==========================================
def optimize_probabilities(G):
    """
    Λύνει το πρόβλημα Ημι-ορισμένου Προγραμματισμού (SDP) 
    για την εύρεση του βέλτιστου πίνακα πιθανοτήτων P.
    """
    n = G.number_of_nodes()
    
    # Η μεταβλητή προς βελτιστοποίηση: Ο συμμετρικός πίνακας πιθανοτήτων P
    P = cp.Variable((n, n), symmetric=True)
    
    constraints = [
        P >= 0, # Οι πιθανότητες δεν μπορούν να είναι αρνητικές
        P @ np.ones(n) == np.ones(n) # Το άθροισμα κάθε γραμμής πρέπει να είναι 1
    ]
    
    # Περιορισμός τοπολογίας: Αν δεν υπάρχει ακμή (i, j), η πιθανότητα είναι 0
    for i in range(n):
        for j in range(n):
            if i != j and not G.has_edge(i, j):
                constraints.append(P[i, j] == 0)
                
    # Ο αναμενόμενος πίνακας W_bar συνδέεται γραμμικά με τον P.
    # Η ελαχιστοποίηση του λ2 του W_bar είναι ισοδύναμη με την ελαχιστοποίηση 
    # της 2ης μεγαλύτερης ιδιοτιμής του P.
    # Αφαιρούμε τον πίνακα J (1/n * 11^T) για να αγνοήσουμε την 1η ιδιοτιμή (που είναι 1).
    J = np.ones((n, n)) / n
    objective = cp.Minimize(cp.norm(P - J, 2))
    
    # Επίλυση του προβλήματος
    prob = cp.Problem(objective, constraints)
    prob.solve(solver=cp.SCS)
    
    return P.value


In [6]:

# ==========================================
# 2. Συνάρτηση για τον Unoptimized Πίνακα 
# ==========================================
def natural_random_walk_probabilities(G):
    """
    Ο unoptimized πίνακας (φυσική τυχαία περιπλάνηση), όπου
    κάθε κόμβος επιλέγει γείτονες ομοιόμορφα τυχαία.
    """
    n = G.number_of_nodes()
    P = np.zeros((n, n))
    for i in range(n):
        neighbors = list(G.neighbors(i))
        deg = len(neighbors)
        for j in neighbors:
            P[i, j] = 1.0 / deg
    return P


In [ ]:

# ==========================================
# 3. Προσομοίωση Ασύγχρονου Gossip (ΔΙΟΡΘΩΜΕΝΟ)
# ==========================================
def simulate_gossip(G, P, initial_values, iterations):
    n = G.number_of_nodes()
    x = np.copy(initial_values)
    x_ave = np.mean(x)
    
    errors = []
    
    for k in range(iterations):
        # 1. Ένας κόμβος 'ξυπνάει' ομοιόμορφα τυχαία
        i = np.random.randint(n)
        
        # 2. Επιλέγει γείτονα με βάση τις πιθανότητες της γραμμής i
        probs = np.copy(P[i, :])
        
        # --- FIX ΓΙΑ ΤΑ NUMERICAL ERRORS ΤΟΥ SDP SOLVER ---
        probs[probs < 0] = 0  # Εξαναγκάζουμε τυχόν αρνητικά (π.χ. -1e-12) να γίνουν 0
        if np.sum(probs) > 0:
            probs = probs / np.sum(probs) # Κανονικοποίηση για να αθροίζουν ακριβώς στο 1
        else:
            # Αν για κάποιο λόγο μηδενίστηκαν όλα, ο κόμβος μιλάει με τον εαυτό του
            probs[i] = 1.0 
        # --------------------------------------------------
            
        j = np.random.choice(n, p=probs)
        
        # 3. Μέσος όρος μεταξύ i και j
        if i != j:
            avg = (x[i] + x[j]) / 2.0
            x[i] = avg
            x[j] = avg
            
        # Υπολογισμός Σφάλματος: l2-νόρμα της απόστασης από τον τέλειο μέσο όρο
        error = np.linalg.norm(x - x_ave * np.ones(n))
        errors.append(error)
        
    return errors


In [8]:

# ==========================================
# 4. Εκτέλεση Πειράματος
# ==========================================
# Δημιουργία ενός Γράφου (π.χ. Path Graph με 10 κόμβους)
# Στους Path graphs η βελτιστοποίηση δείχνει τεράστια διαφορά!
n_nodes = 10
G = nx.path_graph(n_nodes)

# Αρχικές (τυχαίες) τιμές στους αισθητήρες
np.random.seed(42)
initial_values = np.random.rand(n_nodes) * 100

print("Υπολογισμός Unoptimized Πιθανοτήτων...")
P_unopt = natural_random_walk_probabilities(G)

print("Επίλυση SDP για Βελτιστοποιημένες Πιθανότητες (παρακαλώ περιμένετε)...")
P_opt = optimize_probabilities(G)

# Αριθμός βημάτων προσομοίωσης
k_iterations = 3000

print("Προσομοίωση Unoptimized Gossip...")
errors_unopt = simulate_gossip(G, P_unopt, initial_values, k_iterations)

print("Προσομοίωση Optimized Gossip...")
errors_opt = simulate_gossip(G, P_opt, initial_values, k_iterations)

# ==========================================
# 5. Οπτικοποίηση (Plot)
# ==========================================
plt.figure(figsize=(10, 6))
# Χρησιμοποιούμε λογαριθμική κλίμακα στον άξονα Y για να δούμε τον ρυθμό σύγκλισης (exponential decay)
plt.semilogy(errors_unopt, label='Unoptimized Gossip (Natural Random Walk)', alpha=0.8)
plt.semilogy(errors_opt, label='Optimized Gossip (SDP)', linewidth=2)
plt.title(f'Εξέλιξη Σφάλματος Gossip σε Path Graph ({n_nodes} Κόμβοι)')
plt.xlabel('Χρόνος / Επαναλήψεις (k)')
plt.ylabel('Σφάλμα (Λογαριθμική Κλίμακα) - l2 Norm')
plt.legend()
plt.grid(True, which="both", ls="--")
plt.show()

Υπολογισμός Unoptimized Πιθανοτήτων...
Επίλυση SDP για Βελτιστοποιημένες Πιθανότητες (παρακαλώ περιμένετε)...
Προσομοίωση Unoptimized Gossip...
Προσομοίωση Optimized Gossip...


ValueError: probabilities are not non-negative